# 02 — Model Transfer & Structural Diagnostics

Reproduces the main diagnostic experiments for VPN/non-VPN classification using the
**21-feature `safe_core_plus_temporal`** representation:

1. Load audited 21-feature matrix + splits
2. Intra-dataset evaluation (XGBoost, validation-only thresholding, bootstrap CIs)
3. Leave-one-dataset-out (LODO) evaluation with strict no-target-leakage
4. Construction-level scale differences
5. Dataset fingerprinting (predict dataset identity)
6. Feature-effect direction instability (SMD / Cliff's delta / point-biserial / logreg / single-feature AUC)
7. Preprocessing sensitivity audit
8. Robustness checks table
9. Publication-quality figures (300 DPI)
10. Final summary + checklist

**Integrity rule:** no numbers are invented. Anything that cannot be computed from project
files or Notebook 1 outputs is emitted in a clearly marked **MISSING / NEEDS MANUAL INPUT**
cell that names the required file/script. Outputs are saved under `paper_audit_outputs/`.

**21-feature source:** `artifacts/clean_pipeline/features_{train,val,test}.parquet`. These
parquet files contain all 21 `safe_core_plus_temporal` features for **all three** datasets
(including USBVPN, whose per-dataset processed flows are pre-aggregated and lack packet
arrays), together with `dataset`, `label`, `capture_id`, and the canonical capture-level
`split` baked into the filenames. Label encoding: **1 = VPN (positive), 0 = non-VPN**.


In [1]:

# --- Setup: project root, imports, output folders, helpers ---
import os, sys, json, platform, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid"); HAS_SNS = True
except Exception:
    HAS_SNS = False

RNG_SEED = 42
np.random.seed(RNG_SEED)

# Locate project root (dir containing both 'artifacts' and 'data')
ROOT = Path.cwd().resolve()
while not ((ROOT / "artifacts").exists() and (ROOT / "data").exists()) and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("PROJECT ROOT:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT = ROOT / "paper_audit_outputs"
TBL, FIG, MET, LOG = OUT/"tables", OUT/"figures", OUT/"metrics", OUT/"logs"
for d in (TBL, FIG, MET, LOG):
    d.mkdir(parents=True, exist_ok=True)

MISSING = []
def mark_missing(section, what, needed):
    MISSING.append({"section": section, "what": what, "needed": needed})
    print(f"  [MISSING/{section}] {what}  -> needs: {needed}")

GENERATED = []
def save_table(df, name):
    p = TBL / name; df.to_csv(p, index=False)
    GENERATED.append(str(p.relative_to(ROOT)))
    print(f"  saved table: {p.relative_to(ROOT)}  ({df.shape[0]}x{df.shape[1]})"); return p
def save_fig(fig, name, dpi=300):
    p = FIG / name; fig.savefig(p, dpi=dpi, bbox_inches="tight"); plt.close(fig)
    GENERATED.append(str(p.relative_to(ROOT)))
    print(f"  saved figure: {p.relative_to(ROOT)}  (dpi={dpi})"); return p
def save_metric(obj, name):
    p = MET / name; p.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    GENERATED.append(str(p.relative_to(ROOT)))
    print(f"  saved metric: {p.relative_to(ROOT)}"); return p

print("Output folders ready under", OUT.relative_to(ROOT))
print("seaborn:", HAS_SNS)


PROJECT ROOT: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
Output folders ready under paper_audit_outputs
seaborn: True


In [2]:

# --- Metric / threshold / bootstrap helpers (full precision internally) ---
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, accuracy_score,
                             balanced_accuracy_score, precision_score, recall_score, f1_score)

def safe_auc(y, s):
    y = np.asarray(y)
    if len(np.unique(y)) < 2: return np.nan
    return float(roc_auc_score(y, s))

def safe_pr_auc(y, s):
    y = np.asarray(y)
    if len(np.unique(y)) < 2: return np.nan
    return float(average_precision_score(y, s))

def youden_threshold(y_val, s_val):
    """Threshold maximizing Youden's J (tpr - fpr) on VALIDATION only."""
    y_val = np.asarray(y_val)
    if len(np.unique(y_val)) < 2:
        return 0.5  # cannot select on single-class validation; documented fallback
    fpr, tpr, thr = roc_curve(y_val, s_val)
    j = tpr - fpr
    return float(thr[int(np.argmax(j))])

def metrics_at_threshold(y, s, thr):
    y = np.asarray(y); pred = (np.asarray(s) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    fpr = fp/(fp+tn) if (fp+tn) else np.nan
    fnr = fn/(fn+tp) if (fn+tp) else np.nan
    return {
        "roc_auc": safe_auc(y, s), "pr_auc": safe_pr_auc(y, s),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "fpr": float(fpr) if fpr==fpr else None,
        "fnr": float(fnr) if fnr==fnr else None,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "threshold": float(thr),
        "n_pos": int((y==1).sum()), "n_neg": int((y==0).sum()),
    }

def bootstrap_ci(y, s, fn, n=1000, seed=RNG_SEED, alpha=0.05):
    y = np.asarray(y); s = np.asarray(s)
    if len(np.unique(y)) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed); idx = np.arange(len(y)); vals = []
    for _ in range(n):
        bi = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y[bi])) < 2: continue
        vals.append(fn(y[bi], s[bi]))
    if not vals: return (np.nan, np.nan)
    return (float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2))))

def cliffs_delta(a, b):
    """Cliff's delta of group a (VPN) vs b (nonVPN). >0 => a tends larger."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    if len(a)==0 or len(b)==0: return np.nan
    # rank-based O(n log n) computation
    na, nb = len(a), len(b)
    allv = np.concatenate([a, b])
    order = np.argsort(allv, kind="mergesort")
    ranks = np.empty(len(allv)); ranks[order] = np.arange(1, len(allv)+1)
    # average ranks for ties
    s = pd.Series(allv); avg = s.rank(method="average").values
    Ra = avg[:na].sum()
    U = Ra - na*(na+1)/2.0
    return float((2*U)/(na*nb) - 1.0)

print("helpers ready")


helpers ready


## 1. Load audited data and splits

In [3]:

# --- 1a. Load 21-feature matrix (all 3 datasets) from clean_pipeline features_*.parquet ---
FEATURES21 = ["total_packets","total_bytes","mean_pkt_len","std_pkt_len","median_pkt_len",
              "p25_pkt_len","p75_pkt_len","max_pkt_len","min_pkt_len","pkt_len_cv","pkt_len_iqr",
              "iat_mean","iat_std","iat_median","iat_p25","iat_p75","iat_iqr","iat_cv",
              "flow_duration","packet_rate","byte_rate"]
CONSTRUCTION5 = ["flow_duration","total_packets","total_bytes","packet_rate","byte_rate"]
DATASETS = ["iscx","usbvpn","vnat"]
DS_LABEL = {"iscx":"ISCXVPN2016","usbvpn":"USBVPN","vnat":"VNAT"}

src_dir = ROOT/"artifacts"/"clean_pipeline"
frames = []
for sp in ["train","val","test"]:
    p = src_dir/f"features_{sp}.parquet"
    if not p.exists():
        mark_missing("1", f"features_{sp}.parquet not found", str(p)); continue
    df = pd.read_parquet(p); df["split"] = sp; frames.append(df)
assert frames, "no feature parquet loaded"
DATA = pd.concat(frames, ignore_index=True)
missing_feats = [f for f in FEATURES21 if f not in DATA.columns]
assert not missing_feats, f"missing features: {missing_feats}"

# clean: +/-inf -> NaN -> 0.0 (same policy as runtime)
DATA[FEATURES21] = DATA[FEATURES21].replace([np.inf,-np.inf], np.nan).fillna(0.0)
DATA["dataset"] = DATA["dataset"].astype(str)
DATA["label"] = DATA["label"].astype(int)
print("DATA shape:", DATA.shape)
print("label encoding -> 1=VPN, 0=nonVPN")
print(DATA.groupby(["dataset","split"])["label"].value_counts().unstack(fill_value=0))


DATA shape: (72612, 32)
label encoding -> 1=VPN, 0=nonVPN
label              0     1
dataset split             
iscx    test    1328   442
        train   6201  2061
        val     1329   440
usbvpn  test      51  1260
        train  44161  5914
        val       36  1282
vnat    test    1160    55
        train   5413   262
        val     1160    57


In [4]:

# --- 1b. Composition + capture counts per dataset/split ---
rows = []
for ds in DATASETS:
    for sp in ["train","val","test"]:
        sub = DATA[(DATA.dataset==ds)&(DATA.split==sp)]
        rows.append({"dataset":ds,"split":sp,"flows":len(sub),
                     "vpn":int((sub.label==1).sum()),"nonvpn":int((sub.label==0).sum()),
                     "captures":int(sub.capture_id.nunique())})
comp = pd.DataFrame(rows)
save_table(comp, "nb2_data_composition.csv")
comp


  saved table: paper_audit_outputs\tables\nb2_data_composition.csv  (9x6)


,dataset,split,flows,vpn,nonvpn,captures
0,iscx,train,8262,2061,6201,124
1,iscx,val,1769,440,1329,8
2,iscx,test,1770,442,1328,8
3,usbvpn,train,50075,5914,44161,18
4,usbvpn,val,1318,1282,36,4
5,usbvpn,test,1311,1260,51,13
6,vnat,train,5675,262,5413,102
7,vnat,val,1217,57,1160,4
8,vnat,test,1215,55,1160,59


In [5]:

# --- 1c. Leakage verification: capture overlap across splits within each dataset, and global ---
leak_rows = []
for ds in DATASETS + ["__ALL__"]:
    d = DATA if ds=="__ALL__" else DATA[DATA.dataset==ds]
    ctr = set(d[d.split=="train"].capture_id); cva = set(d[d.split=="val"].capture_id); cte = set(d[d.split=="test"].capture_id)
    leak_rows.append({"scope":ds,"train_caps":len(ctr),"val_caps":len(cva),"test_caps":len(cte),
                      "train_val_overlap":len(ctr&cva),"train_test_overlap":len(ctr&cte),
                      "val_test_overlap":len(cva&cte)})
leak = pd.DataFrame(leak_rows)
leak_ok = bool((leak[["train_val_overlap","train_test_overlap","val_test_overlap"]].values==0).all())
print("Within-split capture leakage = ZERO:", leak_ok)
assert leak_ok, "capture overlap detected across splits!"

# LODO leakage principle restated + verified structurally below in Section 3:
# for each target, training data is filtered to dataset != target, so target captures
# can never enter train/val/threshold/scaler fitting. (asserted per-fold in Section 3)
save_table(leak, "nb2_split_leakage_check.csv")
leak


Within-split capture leakage = ZERO: True
  saved table: paper_audit_outputs\tables\nb2_split_leakage_check.csv  (4x7)


,scope,train_caps,val_caps,test_caps,train_val_overlap,train_test_overlap,val_test_overlap
0,iscx,124,8,8,0,0,0
1,usbvpn,18,4,13,0,0,0
2,vnat,102,4,59,0,0,0
3,__ALL__,244,16,80,0,0,0


## 2. Intra-dataset evaluation

Fixed model: `XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
eval_metric="logloss", random_state=42, n_jobs=0, tree_method="hist")`; all other
hyperparameters are XGBoost library defaults (subsample=1.0, colsample_bytree=1.0,
min_child_weight=1, gamma=0.0, reg_lambda=1.0, reg_alpha=0.0, scale_pos_weight=1.0 i.e.
no class weighting, native np.nan missing-value handling; xgboost 3.2.0). Threshold is
selected **only on validation
captures** (Youden's J = argmax(TPR−FPR)), then applied unchanged to the test split.
Test data is never used for thresholding. Bootstrap 95% CIs (1000 flow-level resamples)
are reported for ROC-AUC and PR-AUC because no repeated seeds/splits are available.


In [6]:

import xgboost as xgb
def make_xgb():
    return xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                             eval_metric="logloss", random_state=RNG_SEED,
                             n_jobs=0, tree_method="hist")

intra_rows = []; intra_curves = {}; intra_test_auc = {}; intra_test_prauc = {}
DIAG_MODELS_DIR = ROOT/"artifacts"/"diagnostic_models"; DIAG_MODELS_DIR.mkdir(parents=True, exist_ok=True)
for ds in DATASETS:
    tr = DATA[(DATA.dataset==ds)&(DATA.split=="train")]
    va = DATA[(DATA.dataset==ds)&(DATA.split=="val")]
    te = DATA[(DATA.dataset==ds)&(DATA.split=="test")]
    if tr.label.nunique() < 2:
        mark_missing("2", f"{ds} train is single-class; cannot fit", "more balanced capture split"); continue
    clf = make_xgb(); clf.fit(tr[FEATURES21], tr.label)
    clf.save_model(str(DIAG_MODELS_DIR/f"intra_{ds}.json"))
    s_va = clf.predict_proba(va[FEATURES21])[:,1]
    s_te = clf.predict_proba(te[FEATURES21])[:,1]
    thr = youden_threshold(va.label.values, s_va)
    if va.label.nunique()<2:
        mark_missing("2", f"{ds} validation single-class -> threshold fell back to 0.5", "balanced val captures")
    m = metrics_at_threshold(te.label.values, s_te, thr)
    lo_roc, hi_roc = bootstrap_ci(te.label.values, s_te, safe_auc)
    lo_pr, hi_pr = bootstrap_ci(te.label.values, s_te, safe_pr_auc)
    m.update({"dataset":ds,"roc_auc_ci_lo":lo_roc,"roc_auc_ci_hi":hi_roc,
              "pr_auc_ci_lo":lo_pr,"pr_auc_ci_hi":hi_pr})
    intra_rows.append(m)
    intra_test_auc[ds] = m["roc_auc"]; intra_test_prauc[ds] = m["pr_auc"]
    intra_curves[ds] = {"y":te.label.values,"s":s_te}
    print(f"{ds:7s} test ROC-AUC={m['roc_auc']:.5f} PR-AUC={m['pr_auc']:.5f} thr={thr:.4f} "
          f"recall={m['recall']:.3f} fpr={m['fpr']}")
intra_df = pd.DataFrame(intra_rows)
col_order = ["dataset","roc_auc","roc_auc_ci_lo","roc_auc_ci_hi","pr_auc","pr_auc_ci_lo","pr_auc_ci_hi",
             "accuracy","balanced_accuracy","precision","recall","f1","fpr","fnr",
             "threshold","tn","fp","fn","tp","n_pos","n_neg"]
intra_df = intra_df[[c for c in col_order if c in intra_df.columns]]
save_table(intra_df, "intra_dataset_metrics.csv")
intra_df


iscx    test ROC-AUC=0.98344 PR-AUC=0.97440 thr=0.0568 recall=0.930 fpr=0.030120481927710843


usbvpn  test ROC-AUC=0.98243 PR-AUC=0.99870 thr=0.9972 recall=0.191 fpr=0.0


vnat    test ROC-AUC=1.00000 PR-AUC=1.00000 thr=0.9841 recall=0.382 fpr=0.0
  saved table: paper_audit_outputs\tables\intra_dataset_metrics.csv  (3x21)


,dataset,roc_auc,roc_auc_ci_lo,roc_auc_ci_hi,pr_auc,pr_auc_ci_lo,pr_auc_ci_hi,accuracy,balanced_accuracy,precision,...,f1,fpr,fnr,threshold,tn,fp,fn,tp,n_pos,n_neg
0,iscx,0.983444,0.975662,0.990117,0.974399,0.964104,0.983168,0.959887,0.949872,0.911308,...,0.920493,0.03012,0.070136,0.056794,1288,40,31,411,442,1328
1,usbvpn,0.982431,0.942934,0.999859,0.998698,0.995921,0.999994,0.222731,0.595635,1.000000,...,0.321119,0.00000,0.808730,0.997239,51,0,1019,241,1260,51
2,vnat,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.972016,0.690909,1.000000,...,0.552632,0.00000,0.618182,0.984086,1160,0,34,21,55,1160


In [7]:

# --- 2b. ROC + PR curves, confusion matrices (intra-dataset) ---
fig, ax = plt.subplots(1,2, figsize=(12,5))
for ds in intra_curves:
    y,s = intra_curves[ds]["y"], intra_curves[ds]["s"]
    fpr,tpr,_ = roc_curve(y,s); ax[0].plot(fpr,tpr,label=f"{DS_LABEL[ds]} (AUC={safe_auc(y,s):.3f})")
    pr,rc,_ = precision_recall_curve(y,s); ax[1].plot(rc,pr,label=f"{DS_LABEL[ds]} (AP={safe_pr_auc(y,s):.3f})")
ax[0].plot([0,1],[0,1],"k--",lw=0.7); ax[0].set(title="Intra-dataset ROC",xlabel="FPR",ylabel="TPR"); ax[0].legend()
ax[1].set(title="Intra-dataset PR",xlabel="Recall",ylabel="Precision"); ax[1].legend()
save_fig(fig, "intra_roc_pr_curves.png")

fig, axes = plt.subplots(1, len(intra_curves), figsize=(4*len(intra_curves),3.5))
if len(intra_curves)==1: axes=[axes]
for axc, ds in zip(axes, intra_curves):
    row = intra_df[intra_df.dataset==ds].iloc[0]
    cm = np.array([[row.tn,row.fp],[row.fn,row.tp]])
    im=axc.imshow(cm, cmap="Blues")
    for (i,j),v in np.ndenumerate(cm): axc.text(j,i,int(v),ha="center",va="center")
    axc.set(title=DS_LABEL[ds],xticks=[0,1],yticks=[0,1],
            xticklabels=["nonVPN","VPN"],yticklabels=["nonVPN","VPN"],xlabel="pred",ylabel="true")
fig.suptitle("Intra-dataset confusion matrices (val-selected threshold)")
save_fig(fig, "intra_confusion_matrices.png")


  saved figure: paper_audit_outputs\figures\intra_roc_pr_curves.png  (dpi=300)


  saved figure: paper_audit_outputs\figures\intra_confusion_matrices.png  (dpi=300)


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/figures/intra_confusion_matrices.png')

## 3. Leave-one-dataset-out (LODO) evaluation

For each target dataset, the model trains/validates on the **other two** datasets only.
Source train rows (`split==train`) fit the model; source validation rows (`split==val`)
select the threshold (Youden's J). The **entire** target dataset (all splits) is the held-out
test set. **No target-domain data** touches training, thresholding, scaling, calibration,
or model selection — asserted per fold.


In [8]:

lodo_rows = []; lodo_curves = {}
DIAG_MODELS_DIR = ROOT/"artifacts"/"diagnostic_models"; DIAG_MODELS_DIR.mkdir(parents=True, exist_ok=True)
for target in DATASETS:
    src = DATA[DATA.dataset != target]
    tr = src[src.split=="train"]; va = src[src.split=="val"]
    te = DATA[DATA.dataset == target]            # full target dataset = held-out test
    # strict leakage assertion: no target captures in any source partition
    assert target not in set(tr.dataset) and target not in set(va.dataset)
    assert len(set(te.capture_id) & set(pd.concat([tr,va]).capture_id)) == 0
    if tr.label.nunique() < 2:
        mark_missing("3", f"LODO target {target}: source train single-class", "balanced source"); continue
    clf = make_xgb(); clf.fit(tr[FEATURES21], tr.label)
    clf.save_model(str(DIAG_MODELS_DIR/f"lodo_target_{target}.json"))
    s_va = clf.predict_proba(va[FEATURES21])[:,1]
    s_te = clf.predict_proba(te[FEATURES21])[:,1]
    thr = youden_threshold(va.label.values, s_va)
    m = metrics_at_threshold(te.label.values, s_te, thr)
    m.update({"target":target, "train_datasets":"+".join([d for d in DATASETS if d!=target]),
              "src_train_pos":int((tr.label==1).sum()),"src_train_neg":int((tr.label==0).sum()),
              "src_val_pos":int((va.label==1).sum()),"src_val_neg":int((va.label==0).sum()),
              "test_vpn":int((te.label==1).sum()),"test_nonvpn":int((te.label==0).sum()),
              "test_captures":int(te.capture_id.nunique()),
              "intra_roc_auc":intra_test_auc.get(target,np.nan),
              "intra_pr_auc":intra_test_prauc.get(target,np.nan)})
    m["roc_auc_gap"] = (m["intra_roc_auc"] - m["roc_auc"]) if m["intra_roc_auc"]==m["intra_roc_auc"] else np.nan
    m["pr_auc_gap"]  = (m["intra_pr_auc"]  - m["pr_auc"])  if m["intra_pr_auc"]==m["intra_pr_auc"] else np.nan
    lodo_rows.append(m); lodo_curves[target] = {"y":te.label.values,"s":s_te}
    print(f"target {target:7s} LODO ROC-AUC={m['roc_auc']:.5f} PR-AUC={m['pr_auc']:.5f} "
          f"gap_roc={m['roc_auc_gap']:.4f} test(vpn={m['test_vpn']},non={m['test_nonvpn']})")
lodo_df = pd.DataFrame(lodo_rows)
lc = ["target","train_datasets","roc_auc","pr_auc","intra_roc_auc","intra_pr_auc","roc_auc_gap","pr_auc_gap",
      "accuracy","balanced_accuracy","precision","recall","f1","fpr","fnr","threshold",
      "test_vpn","test_nonvpn","test_captures","src_train_pos","src_train_neg","src_val_pos","src_val_neg",
      "tn","fp","fn","tp"]
lodo_df = lodo_df[[c for c in lc if c in lodo_df.columns]]
save_table(lodo_df, "lodo_metrics.csv")
lodo_roc = {r["target"]:r["roc_auc"] for _,r in lodo_df.iterrows()}
lodo_pr  = {r["target"]:r["pr_auc"] for _,r in lodo_df.iterrows()}
print("LODO mean ROC-AUC:", np.nanmean(list(lodo_roc.values())),
      "| LODO min:", np.nanmin(list(lodo_roc.values())))
lodo_df


target iscx    LODO ROC-AUC=0.48498 PR-AUC=0.28348 gap_roc=0.4985 test(vpn=2943,non=8858)


target usbvpn  LODO ROC-AUC=0.54823 PR-AUC=0.18762 gap_roc=0.4342 test(vpn=8456,non=44248)


target vnat    LODO ROC-AUC=0.71297 PR-AUC=0.10956 gap_roc=0.2870 test(vpn=374,non=7733)
  saved table: paper_audit_outputs\tables\lodo_metrics.csv  (3x27)
LODO mean ROC-AUC: 0.5820630243513644 | LODO min: 0.484981871636966


,target,train_datasets,roc_auc,pr_auc,intra_roc_auc,intra_pr_auc,roc_auc_gap,pr_auc_gap,accuracy,balanced_accuracy,...,test_nonvpn,test_captures,src_train_pos,src_train_neg,src_val_pos,src_val_neg,tn,fp,fn,tp
0,iscx,usbvpn+vnat,0.484982,0.283483,0.983444,0.974399,0.498462,0.690915,0.553682,0.468654,...,8858,140,6176,49574,1339,1196,5654,3204,2063,880
1,usbvpn,iscx+vnat,0.548234,0.187624,0.982431,0.998698,0.434197,0.811074,0.502543,0.541548,...,44248,35,2323,11614,497,2489,21421,22827,3391,5065
2,vnat,iscx+usbvpn,0.712974,0.109557,1.000000,1.000000,0.287026,0.890443,0.568274,0.759702,...,7733,165,7975,50362,1722,1365,4244,3489,11,363


In [9]:

# --- 3b. intra vs cross bar plots + LODO ROC/PR curves + confusion matrices ---
labels=[DS_LABEL[d] for d in DATASETS]
x=np.arange(len(DATASETS)); w=0.38
fig,ax=plt.subplots(1,2,figsize=(13,5))
ax[0].bar(x-w/2,[intra_test_auc.get(d,np.nan) for d in DATASETS],w,label="intra")
ax[0].bar(x+w/2,[lodo_roc.get(d,np.nan) for d in DATASETS],w,label="cross (LODO)")
ax[0].axhline(0.5,color="k",ls="--",lw=0.7); ax[0].set(title="Intra vs Cross ROC-AUC",xticks=x); ax[0].set_xticklabels(labels); ax[0].legend()
ax[1].bar(x-w/2,[intra_test_prauc.get(d,np.nan) for d in DATASETS],w,label="intra")
ax[1].bar(x+w/2,[lodo_pr.get(d,np.nan) for d in DATASETS],w,label="cross (LODO)")
ax[1].set(title="Intra vs Cross PR-AUC",xticks=x); ax[1].set_xticklabels(labels); ax[1].legend()
save_fig(fig,"intra_vs_cross_auc.png")

fig,ax=plt.subplots(1,2,figsize=(12,5))
for t in lodo_curves:
    y,s=lodo_curves[t]["y"],lodo_curves[t]["s"]
    fpr,tpr,_=roc_curve(y,s); ax[0].plot(fpr,tpr,label=f"{DS_LABEL[t]} (AUC={safe_auc(y,s):.3f})")
    pr,rc,_=precision_recall_curve(y,s); ax[1].plot(rc,pr,label=f"{DS_LABEL[t]} (AP={safe_pr_auc(y,s):.3f})")
ax[0].plot([0,1],[0,1],"k--",lw=0.7); ax[0].set(title="LODO ROC",xlabel="FPR",ylabel="TPR"); ax[0].legend()
ax[1].set(title="LODO PR",xlabel="Recall",ylabel="Precision"); ax[1].legend()
save_fig(fig,"lodo_roc_pr_curves.png")

fig,axes=plt.subplots(1,len(lodo_df),figsize=(4*len(lodo_df),3.5))
if len(lodo_df)==1: axes=[axes]
for axc,(_,row) in zip(axes,lodo_df.iterrows()):
    cm=np.array([[row.tn,row.fp],[row.fn,row.tp]]); axc.imshow(cm,cmap="Oranges")
    for (i,j),v in np.ndenumerate(cm): axc.text(j,i,int(v),ha="center",va="center")
    axc.set(title=f"LODO {DS_LABEL[row.target]}",xticks=[0,1],yticks=[0,1],
            xticklabels=["nonVPN","VPN"],yticklabels=["nonVPN","VPN"],xlabel="pred",ylabel="true")
fig.suptitle("LODO confusion matrices (source-val threshold)")
save_fig(fig,"lodo_confusion_matrices.png")


  saved figure: paper_audit_outputs\figures\intra_vs_cross_auc.png  (dpi=300)


  saved figure: paper_audit_outputs\figures\lodo_roc_pr_curves.png  (dpi=300)


  saved figure: paper_audit_outputs\figures\lodo_confusion_matrices.png  (dpi=300)


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/figures/lodo_confusion_matrices.png')

## 4. Construction-level scale differences

In [10]:

desc = CONSTRUCTION5
rows=[]
for ds in DATASETS:
    sub = DATA[DATA.dataset==ds]
    for f in desc:
        q1,med,q3 = sub[f].quantile([0.25,0.5,0.75])
        rows.append({"dataset":ds,"descriptor":f,"q1":float(q1),"median":float(med),"q3":float(q3)})
scale_df = pd.DataFrame(rows)
save_table(scale_df, "construction_scale_with_iqr.csv")

fig,axes=plt.subplots(1,len(desc),figsize=(4*len(desc),4))
for axc,f in zip(axes,desc):
    data=[np.log1p(DATA[DATA.dataset==ds][f].clip(lower=0).values) for ds in DATASETS]
    axc.boxplot(data,labels=[DS_LABEL[d] for d in DATASETS],showfliers=False)
    axc.set(title=f,ylabel="log1p value"); axc.tick_params(axis="x",rotation=20)
fig.suptitle("Construction-scale descriptors (log1p)")
save_fig(fig,"construction_scale_boxplots.png")
scale_df.pivot(index="descriptor",columns="dataset",values="median")


  saved table: paper_audit_outputs\tables\construction_scale_with_iqr.csv  (15x5)


  saved figure: paper_audit_outputs\figures\construction_scale_boxplots.png  (dpi=300)


dataset,iscx,usbvpn,vnat
descriptor,,,
byte_rate,105.841040,150380.219089,2302.651457
flow_duration,12.553336,0.119148,4.410715
packet_rate,0.727865,322.571300,8.444046
total_bytes,1200.000000,12417.000000,279.000000
total_packets,8.000000,30.000000,4.000000


## 5. Dataset fingerprinting audit

Predict **dataset identity** (3-class) to quantify how separable the datasets are by
construction. Capture-level split reused from the `split` column (train→fit, test→evaluate).
**Experiment A** uses 5 construction descriptors; **Experiment B** uses all 21 features.

**Protocol (reproducibility).** Dataset-fingerprinting classifiers were evaluated using
capture-level train/test partitioning: all flows from a given capture are assigned to a
single split, so no capture is split across train and test (zero capture overlap is
asserted in §1c). Dataset labels are predicted only on **held-out test captures** that are
disjoint from the captures used for fitting; the validation split is not used here. The
classifier is an XGBoost multiclass model, `XGBClassifier(n_estimators=300, max_depth=5,
learning_rate=0.1, objective="multi:softprob", num_class=3, eval_metric="mlogloss",
random_state=42, n_jobs=0, tree_method="hist")`, with all other hyperparameters left at
XGBoost 3.2.0 library defaults (native np.nan missing-value handling). This is a single
capture-level held-out split rather than k-fold cross-validation; sampling uncertainty is
instead reported as a 500-resample flow-level bootstrap 95\% CI on macro-AUC. No class
imbalance correction (resampling or class weighting) is applied; we report **macro**-averaged
AUC and F1 so each dataset contributes equally regardless of its flow count. Because the
train/test captures are disjoint, the reported separability is not a within-capture leakage
artifact but reflects genuine cross-dataset structural shift.


In [11]:

from sklearn.preprocessing import label_binarize
ds_to_int = {d:i for i,d in enumerate(DATASETS)}
def fingerprint(feature_cols, tag):
    tr = DATA[DATA.split=="train"]; te = DATA[DATA.split=="test"]
    ytr = tr.dataset.map(ds_to_int).values; yte = te.dataset.map(ds_to_int).values
    clf = xgb.XGBClassifier(n_estimators=300,max_depth=5,learning_rate=0.1,
                            objective="multi:softprob",num_class=len(DATASETS),
                            eval_metric="mlogloss",random_state=RNG_SEED,n_jobs=0,tree_method="hist")
    clf.fit(tr[feature_cols], ytr)
    proba = clf.predict_proba(te[feature_cols]); pred = proba.argmax(1)
    Yb = label_binarize(yte, classes=list(range(len(DATASETS))))
    macro_auc = float(roc_auc_score(Yb, proba, average="macro", multi_class="ovr"))
    per_class = {DATASETS[i]: float(roc_auc_score(Yb[:,i], proba[:,i])) for i in range(len(DATASETS))}
    acc=float(accuracy_score(yte,pred)); mf1=float(f1_score(yte,pred,average="macro"))
    cm=confusion_matrix(yte,pred,labels=list(range(len(DATASETS))))
    # bootstrap CI for macro-AUC
    rng=np.random.default_rng(RNG_SEED); idx=np.arange(len(yte)); vals=[]
    for _ in range(500):
        bi=rng.choice(idx,len(idx),replace=True)
        if len(np.unique(yte[bi]))<len(DATASETS): continue
        try: vals.append(roc_auc_score(Yb[bi],proba[bi],average="macro",multi_class="ovr"))
        except Exception: pass
    ci=(float(np.percentile(vals,2.5)),float(np.percentile(vals,97.5))) if vals else (np.nan,np.nan)
    return {"experiment":tag,"n_features":len(feature_cols),"macro_auc":macro_auc,
            "macro_auc_ci_lo":ci[0],"macro_auc_ci_hi":ci[1],"accuracy":acc,"macro_f1":mf1,
            **{f"auc_{DATASETS[i]}":per_class[DATASETS[i]] for i in range(len(DATASETS))}}, cm, clf, feature_cols

fpA, cmA, clfA, colsA = fingerprint(CONSTRUCTION5, "A_construction5")
fpB, cmB, clfB, colsB = fingerprint(FEATURES21, "B_full21")
fp_df = pd.DataFrame([fpA, fpB])
save_table(fp_df, "dataset_fingerprinting_metrics.csv")
print(fp_df[["experiment","n_features","macro_auc","accuracy","macro_f1"]].to_string(index=False))
fp_df


  saved table: paper_audit_outputs\tables\dataset_fingerprinting_metrics.csv  (2x10)
     experiment  n_features  macro_auc  accuracy  macro_f1
A_construction5           5   0.985457  0.883380  0.871950
       B_full21          21   0.999316  0.972765  0.972784


,experiment,n_features,macro_auc,macro_auc_ci_lo,macro_auc_ci_hi,accuracy,macro_f1,auc_iscx,auc_usbvpn,auc_vnat
0,A_construction5,5,0.985457,0.983645,0.987148,0.883380,0.871950,0.988406,0.993450,0.974516
1,B_full21,21,0.999316,0.999067,0.999515,0.972765,0.972784,0.998462,0.999941,0.999546


In [12]:

# --- 5b. fingerprinting confusion matrices, OvR ROC, feature importance (Exp B) ---
fig,axes=plt.subplots(1,2,figsize=(9,4))
for axc,(cm,tag) in zip(axes,[(cmA,"A: construction-5"),(cmB,"B: full-21")]):
    axc.imshow(cm,cmap="Purples")
    for (i,j),v in np.ndenumerate(cm): axc.text(j,i,int(v),ha="center",va="center",fontsize=8)
    axc.set(title=tag,xticks=range(len(DATASETS)),yticks=range(len(DATASETS)),
            xticklabels=[DS_LABEL[d] for d in DATASETS],yticklabels=[DS_LABEL[d] for d in DATASETS],
            xlabel="pred",ylabel="true")
    axc.tick_params(axis="x",rotation=25)
fig.suptitle("Dataset fingerprinting confusion matrices")
save_fig(fig,"dataset_fingerprinting_confusion.png")

# OvR ROC for Exp B
te=DATA[DATA.split=="test"]; yte=te.dataset.map(ds_to_int).values
proba=clfB.predict_proba(te[FEATURES21]); Yb=label_binarize(yte,classes=list(range(len(DATASETS))))
fig,ax=plt.subplots(figsize=(6,5))
for i,d in enumerate(DATASETS):
    fpr,tpr,_=roc_curve(Yb[:,i],proba[:,i]); ax.plot(fpr,tpr,label=f"{DS_LABEL[d]} (AUC={roc_auc_score(Yb[:,i],proba[:,i]):.3f})")
ax.plot([0,1],[0,1],"k--",lw=0.7); ax.set(title="Fingerprinting OvR ROC (full-21)",xlabel="FPR",ylabel="TPR"); ax.legend()
save_fig(fig,"dataset_fingerprinting_ovr_roc.png")

imp=pd.DataFrame({"feature":FEATURES21,"importance":clfB.feature_importances_}).sort_values("importance",ascending=False)
fig,ax=plt.subplots(figsize=(7,6)); ax.barh(imp.feature[::-1],imp.importance[::-1]); ax.set(title="Fingerprinting feature importance (full-21)")
save_fig(fig,"dataset_fingerprinting_importance.png")
save_table(imp,"dataset_fingerprinting_importance.csv")


  saved figure: paper_audit_outputs\figures\dataset_fingerprinting_confusion.png  (dpi=300)


  saved figure: paper_audit_outputs\figures\dataset_fingerprinting_ovr_roc.png  (dpi=300)


  saved figure: paper_audit_outputs\figures\dataset_fingerprinting_importance.png  (dpi=300)
  saved table: paper_audit_outputs\tables\dataset_fingerprinting_importance.csv  (21x2)


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/tables/dataset_fingerprinting_importance.csv')

## 6. Feature-effect direction instability

For each of the 21 features and each dataset, the VPN(=1) vs non-VPN(=0) direction is
measured with six signals: raw mean diff, raw median diff, standardized mean difference
(**SMD** = (mean_VPN − mean_nonVPN)/pooled_std, ddof=0), Cliff's delta, point-biserial
correlation sign, single-feature logistic-regression coefficient sign, and single-feature
ROC-AUC direction (AUC−0.5).

**Classification rule:** a feature is flagged **verified raw-space direction instability**
only if its **SMD signs differ across datasets**. If SMD signs agree but rank-based signals
(Cliff's delta / single-feature AUC) disagree, it is labelled **rank-based direction
instability**. Otherwise **stable**.


In [13]:

from scipy.stats import pointbiserialr
from sklearn.linear_model import LogisticRegression
def sign(x):
    if x!=x: return 0
    return 1 if x>0 else (-1 if x<0 else 0)

inst_rows=[]; metric_matrix_rows=[]
for f in FEATURES21:
    smd={}; signs={}; per_ds_metrics={}
    for ds in DATASETS:
        sub=DATA[DATA.dataset==ds]
        a=sub[sub.label==1][f].values; b=sub[sub.label==0][f].values
        if len(a)<2 or len(b)<2:
            smd[ds]=np.nan; signs[ds]=0; per_ds_metrics[ds]={}; continue
        mean_diff=a.mean()-b.mean(); med_diff=np.median(a)-np.median(b)
        pooled=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/max(len(a)+len(b)-2,1))
        s=mean_diff/pooled if pooled>0 else 0.0
        cd=cliffs_delta(a,b)
        try: pb=pointbiserialr(sub.label.values, sub[f].values)[0]
        except Exception: pb=np.nan
        x=sub[[f]].values.astype(float); x=(x-x.mean())/(x.std()+1e-12)
        try:
            lr=LogisticRegression(max_iter=200).fit(x,sub.label.values); coef=float(lr.coef_[0,0])
        except Exception: coef=np.nan
        fa=safe_auc(sub.label.values, sub[f].values); auc_dir=(fa-0.5) if fa==fa else np.nan
        smd[ds]=s; signs[ds]=sign(s)
        per_ds_metrics[ds]={"mean_diff":mean_diff,"med_diff":med_diff,"smd":s,"cliffs":cd,
                            "pointbiserial":pb,"logreg_coef":coef,"auc_dir":auc_dir}
        metric_matrix_rows.append({"feature":f,"dataset":ds,"mean_diff_sign":sign(mean_diff),
            "median_diff_sign":sign(med_diff),"smd_sign":sign(s),"cliffs_sign":sign(cd),
            "pointbiserial_sign":sign(pb),"logreg_sign":sign(coef),"auc_dir_sign":sign(auc_dir)})
    valid_smd_signs={signs[d] for d in DATASETS if smd[d]==smd[d] and signs[d]!=0}
    smd_flip = len(valid_smd_signs)>1
    # rank-based instability: AUC-direction or Cliff sign disagreement across datasets
    rank_signs=set()
    for ds in DATASETS:
        m=per_ds_metrics.get(ds,{})
        if m and m.get("auc_dir")==m.get("auc_dir") and sign(m.get("auc_dir"))!=0:
            rank_signs.add(sign(m["auc_dir"]))
    rank_flip = len(rank_signs)>1
    n_support = sum([smd_flip, rank_flip])
    if smd_flip:
        verdict="verified raw-space direction instability"
    elif rank_flip:
        verdict="rank-based direction instability"
    else:
        verdict="stable feature"
    inst_rows.append({"feature":f,
        "iscx_smd":smd.get("iscx"),"usbvpn_smd":smd.get("usbvpn"),"vnat_smd":smd.get("vnat"),
        "iscx_sign":signs.get("iscx"),"usbvpn_sign":signs.get("usbvpn"),"vnat_sign":signs.get("vnat"),
        "smd_sign_flip":smd_flip,"rank_sign_flip":rank_flip,
        "n_metrics_supporting_instability":n_support,
        "instability_type":verdict,"final_verdict":verdict})
inst_df=pd.DataFrame(inst_rows)
save_table(inst_df, "feature_instability_detailed.csv")
metric_matrix=pd.DataFrame(metric_matrix_rows)
save_table(metric_matrix, "feature_instability_by_metric_matrix.csv")
print(inst_df["final_verdict"].value_counts().to_dict())
inst_df[["feature","iscx_smd","usbvpn_smd","vnat_smd","instability_type"]]


  saved table: paper_audit_outputs\tables\feature_instability_detailed.csv  (21x12)
  saved table: paper_audit_outputs\tables\feature_instability_by_metric_matrix.csv  (63x9)
{'verified raw-space direction instability': 16, 'rank-based direction instability': 5}


,feature,iscx_smd,usbvpn_smd,vnat_smd,instability_type
0,total_packets,0.420977,1.608497,0.329497,rank-based direction instability
1,total_bytes,0.112946,1.350662,0.018129,rank-based direction instability
2,mean_pkt_len,-0.016055,1.025253,-0.351143,verified raw-space direction instability
3,std_pkt_len,0.010386,0.140892,-0.595888,verified raw-space direction instability
4,median_pkt_len,-0.047464,1.315923,-0.150514,verified raw-space direction instability
5,p25_pkt_len,-0.405542,1.473675,0.349444,verified raw-space direction instability
6,p75_pkt_len,0.081877,0.720418,-0.278412,verified raw-space direction instability
7,max_pkt_len,-0.093898,0.272905,-0.504692,verified raw-space direction instability
8,min_pkt_len,-0.733595,1.110173,0.968942,verified raw-space direction instability
9,pkt_len_cv,0.327737,-1.261927,-0.806791,verified raw-space direction instability


In [14]:

# --- 6b. instability verdict category listing + SMD heatmap + by-metric matrix figure ---
cat_rows=[]
for verdict, grp in inst_df.groupby("final_verdict"):
    cat_rows.append({"category":verdict,"n_features":len(grp),"features":", ".join(grp.feature)})
cat_df=pd.DataFrame(cat_rows); save_table(cat_df,"instability_verdict_features.csv")

smd_mat=inst_df.set_index("feature")[["iscx_smd","usbvpn_smd","vnat_smd"]]
fig,ax=plt.subplots(figsize=(6,9))
vmax=np.nanmax(np.abs(smd_mat.values)); im=ax.imshow(smd_mat.values,cmap="coolwarm",vmin=-vmax,vmax=vmax,aspect="auto")
ax.set(xticks=range(3),xticklabels=["ISCX","USBVPN","VNAT"],yticks=range(len(smd_mat)),yticklabels=smd_mat.index)
for (i,j),v in np.ndenumerate(smd_mat.values):
    if v==v: ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=7)
fig.colorbar(im,ax=ax,label="SMD (VPN - nonVPN)"); ax.set_title("SMD heatmap (sign flip = instability)")
save_fig(fig,"smd_heatmap.png")

# by-metric sign instability matrix: for each feature, count distinct signs per metric across datasets
sign_cols=["mean_diff_sign","median_diff_sign","smd_sign","cliffs_sign","pointbiserial_sign","logreg_sign","auc_dir_sign"]
flip_mat=[]
for f in FEATURES21:
    sub=metric_matrix[metric_matrix.feature==f]; row=[]
    for c in sign_cols:
        s={v for v in sub[c].values if v!=0}; row.append(1 if len(s)>1 else 0)
    flip_mat.append(row)
flip_mat=np.array(flip_mat)
fig,ax=plt.subplots(figsize=(8,9)); ax.imshow(flip_mat,cmap="Reds",aspect="auto",vmin=0,vmax=1)
ax.set(xticks=range(len(sign_cols)),xticklabels=[c.replace("_sign","") for c in sign_cols],
       yticks=range(len(FEATURES21)),yticklabels=FEATURES21,title="Direction sign-flip by metric (red=flips across datasets)")
ax.tick_params(axis="x",rotation=40)
save_fig(fig,"instability_by_metric.png")
cat_df


  saved table: paper_audit_outputs\tables\instability_verdict_features.csv  (2x3)


  saved figure: paper_audit_outputs\figures\smd_heatmap.png  (dpi=300)


  saved figure: paper_audit_outputs\figures\instability_by_metric.png  (dpi=300)


,category,n_features,features
0,rank-based direction instability,5,"total_packets, total_bytes, iat_mean, iat_p75,..."
1,verified raw-space direction instability,16,"mean_pkt_len, std_pkt_len, median_pkt_len, p25..."


## 7. Preprocessing sensitivity audit

Re-evaluate per-feature SMD-sign stability under several preprocessing transforms fit on the
pooled feature space. Covariate reweighting and lightweight domain alignment are not present
as reusable project components and are marked **MISSING / NEEDS MANUAL INPUT**.


In [15]:

from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.decomposition import PCA

def smd_signs_for_matrix(X, name):
    """Return dict feature->set of SMD signs across datasets, plus per-(feat,ds) sign."""
    out={}; Xdf=pd.DataFrame(X, columns=[f"c{i}" for i in range(X.shape[1])])
    Xdf["dataset"]=DATA.dataset.values; Xdf["label"]=DATA.label.values
    signs={}
    for ci,f in enumerate(FEATURES21):
        col=f"c{ci}"; ss=set(); per={}
        for ds in DATASETS:
            sub=Xdf[Xdf.dataset==ds]; a=sub[sub.label==1][col].values; b=sub[sub.label==0][col].values
            if len(a)<2 or len(b)<2: per[ds]=0; continue
            pooled=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/max(len(a)+len(b)-2,1))
            s=(a.mean()-b.mean())/pooled if pooled>0 else 0.0
            sg=1 if s>0 else (-1 if s<0 else 0); per[ds]=sg
            if sg!=0: ss.add(sg)
        signs[f]={"flip":len(ss)>1,"per":per}
    return signs

X0=DATA[FEATURES21].values.astype(float)
transforms={}
transforms["raw"]=X0
transforms["log1p"]=np.log1p(np.clip(X0,0,None))
transforms["standard"]=StandardScaler().fit_transform(X0)
transforms["robust"]=RobustScaler().fit_transform(X0)
transforms["quantile"]=QuantileTransformer(output_distribution="normal",random_state=RNG_SEED,
                                            n_quantiles=min(1000,len(X0))).fit_transform(X0)
# whitening = PCA(whiten=True) keeping all comps, then map back is not 1:1 to features;
# we apply standardize+PCA-whiten and re-evaluate in PC space mapped per original feature index
Xs=StandardScaler().fit_transform(X0)
transforms["whitening"]=PCA(whiten=True,random_state=RNG_SEED).fit_transform(Xs)
transforms["pca_projection"]=PCA(n_components=min(10,X0.shape[1]),random_state=RNG_SEED).fit_transform(Xs)

# whitening/pca change the basis: SMD-sign per ORIGINAL feature is not directly defined.
# We therefore report SMD-sign stability only for component-preserving transforms, and
# treat whitening/PCA as basis-changing (flagged separately).
component_preserving=["raw","log1p","standard","robust","quantile"]
sens_rows=[]
sign_cache={}
for name in component_preserving:
    sign_cache[name]=smd_signs_for_matrix(transforms[name], name)
for f in FEATURES21:
    row={"feature":f}
    for name in component_preserving:
        row[name+"_smd_flip"]=int(sign_cache[name][f]["flip"])
    row["n_transforms_with_flip"]=sum(row[name+"_smd_flip"] for name in component_preserving)
    sens_rows.append(row)
sens_df=pd.DataFrame(sens_rows)
save_table(sens_df,"preprocessing_sensitivity.csv")

mark_missing("7","covariate reweighting transform not available as reusable component",
             "a project module/script implementing importance/covariate reweighting")
mark_missing("7","lightweight domain alignment transform not available",
             "a project module/script implementing CORAL/feature alignment")
md_note=("whitening and PCA projection change the feature basis, so a per-original-feature SMD "
         "sign is not well defined for them; they are reported as basis-changing and excluded "
         "from the per-feature sign-flip table.")
print(md_note)
print("features with >=1 transform sign-flip:", int((sens_df.n_transforms_with_flip>0).sum()))
sens_df


  saved table: paper_audit_outputs\tables\preprocessing_sensitivity.csv  (21x7)
  [MISSING/7] covariate reweighting transform not available as reusable component  -> needs: a project module/script implementing importance/covariate reweighting
  [MISSING/7] lightweight domain alignment transform not available  -> needs: a project module/script implementing CORAL/feature alignment
whitening and PCA projection change the feature basis, so a per-original-feature SMD sign is not well defined for them; they are reported as basis-changing and excluded from the per-feature sign-flip table.
features with >=1 transform sign-flip: 21


,feature,raw_smd_flip,log1p_smd_flip,standard_smd_flip,robust_smd_flip,quantile_smd_flip,n_transforms_with_flip
0,total_packets,0,1,0,0,1,2
1,total_bytes,0,1,0,0,1,2
2,mean_pkt_len,1,1,1,1,1,5
3,std_pkt_len,1,1,1,1,1,5
4,median_pkt_len,1,1,1,1,1,5
5,p25_pkt_len,1,1,1,1,1,5
6,p75_pkt_len,1,1,1,1,1,5
7,max_pkt_len,1,1,1,1,1,5
8,min_pkt_len,1,1,1,1,1,5
9,pkt_len_cv,1,1,1,1,1,5


In [16]:

# --- 7b. preprocessing sensitivity heatmaps (color + black/white friendly) ---
mat=sens_df.set_index("feature")[[c+"_smd_flip" for c in component_preserving]]
fig,ax=plt.subplots(figsize=(7,9)); ax.imshow(mat.values,cmap="Reds",vmin=0,vmax=1,aspect="auto")
ax.set(xticks=range(len(component_preserving)),xticklabels=component_preserving,
       yticks=range(len(mat)),yticklabels=mat.index,title="SMD sign-flip under preprocessing (red=flip)")
ax.tick_params(axis="x",rotation=25)
save_fig(fig,"preprocessing_sensitivity_heatmap.png")

fig,ax=plt.subplots(figsize=(7,9)); ax.imshow(mat.values,cmap="gray_r",vmin=0,vmax=1,aspect="auto")
for (i,j),v in np.ndenumerate(mat.values): ax.text(j,i,"X" if v else "",ha="center",va="center",color="white" if v else "black",fontsize=8)
ax.set(xticks=range(len(component_preserving)),xticklabels=component_preserving,
       yticks=range(len(mat)),yticklabels=mat.index,title="SMD sign-flip (B/W friendly, X=flip)")
ax.tick_params(axis="x",rotation=25)
save_fig(fig,"preprocessing_sensitivity_heatmap_bw.png")


  saved figure: paper_audit_outputs\figures\preprocessing_sensitivity_heatmap.png  (dpi=300)


  saved figure: paper_audit_outputs\figures\preprocessing_sensitivity_heatmap_bw.png  (dpi=300)


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/figures/preprocessing_sensitivity_heatmap_bw.png')

## 8. Robustness checks table

Each check is run through the same strict LODO harness (XGBoost, source train fit, source-val
threshold, target held out). We report LODO mean/min ROC-AUC and PR-AUC. Checks that require
components not available in the project are marked **MISSING / NEEDS MANUAL INPUT** rather than
fabricated (covariate reweighting, lightweight alignment, packet-sequence models).


In [17]:

from sklearn.preprocessing import StandardScaler as _SS, QuantileTransformer as _QT, RobustScaler as _RS
from sklearn.decomposition import PCA as _PCA

def lodo_run(feature_cols, transform=None, thr_mode="youden", fit_on="train"):
    """Strict LODO; transform fit on SOURCE only. Returns dict target->(roc,pr) + mean/min."""
    res={}
    for target in DATASETS:
        src=DATA[DATA.dataset!=target]; tr=src[src.split=="train"]; va=src[src.split=="val"]
        te=DATA[DATA.dataset==target]
        if tr.label.nunique()<2: res[target]=(np.nan,np.nan); continue
        Xtr=tr[feature_cols].values.astype(float); Xva=va[feature_cols].values.astype(float); Xte=te[feature_cols].values.astype(float)
        if transform=="log1p":
            Xtr,Xva,Xte=[np.log1p(np.clip(X,0,None)) for X in (Xtr,Xva,Xte)]
        elif transform=="standard":
            sc=_SS().fit(Xtr); Xtr,Xva,Xte=sc.transform(Xtr),sc.transform(Xva),sc.transform(Xte)
        elif transform=="robust":
            sc=_RS().fit(Xtr); Xtr,Xva,Xte=sc.transform(Xtr),sc.transform(Xva),sc.transform(Xte)
        elif transform=="quantile":
            sc=_QT(output_distribution="normal",random_state=RNG_SEED,n_quantiles=min(1000,len(Xtr))).fit(Xtr)
            Xtr,Xva,Xte=sc.transform(Xtr),sc.transform(Xva),sc.transform(Xte)
        elif transform=="whiten":
            sc=_SS().fit(Xtr); Xtr,Xva,Xte=sc.transform(Xtr),sc.transform(Xva),sc.transform(Xte)
            pc=_PCA(whiten=True,random_state=RNG_SEED).fit(Xtr); Xtr,Xva,Xte=pc.transform(Xtr),pc.transform(Xva),pc.transform(Xte)
        elif transform=="pca":
            sc=_SS().fit(Xtr); Xtr,Xva,Xte=sc.transform(Xtr),sc.transform(Xva),sc.transform(Xte)
            pc=_PCA(n_components=min(10,Xtr.shape[1]),random_state=RNG_SEED).fit(Xtr); Xtr,Xva,Xte=pc.transform(Xtr),pc.transform(Xva),pc.transform(Xte)
        clf=make_xgb(); clf.fit(Xtr,tr.label)
        s_va=clf.predict_proba(Xva)[:,1]; s_te=clf.predict_proba(Xte)[:,1]
        res[target]=(safe_auc(te.label.values,s_te), safe_pr_auc(te.label.values,s_te))
    rocs=[v[0] for v in res.values()]; prs=[v[1] for v in res.values()]
    return res, float(np.nanmean(rocs)), float(np.nanmin(rocs)), float(np.nanmean(prs)), float(np.nanmin(prs))

checks=[]
def add_check(setup, features_desc, fcols, transform, observation):
    res,mroc,nroc,mpr,npr=lodo_run(fcols, transform=transform)
    checks.append({"check":setup,"model":"XGBoost(300,5,0.1)","features":features_desc,
        "train_domains":"two source datasets","val_domains":"source val split","test_domains":"held-out dataset",
        "roc_auc_iscx":res["iscx"][0],"roc_auc_usbvpn":res["usbvpn"][0],"roc_auc_vnat":res["vnat"][0],
        "lodo_mean_roc":mroc,"lodo_min_roc":nroc,"lodo_mean_pr":mpr,"lodo_min_pr":npr,
        "observation":observation})

add_check("baseline 21-feature LODO","21 safe_core_plus_temporal",FEATURES21,None,"reference LODO")
add_check("rate-feature removal","21 minus packet_rate,byte_rate",[f for f in FEATURES21 if f not in ("packet_rate","byte_rate")],None,"drop rate features")
add_check("feature pruning (no construction descriptors)","21 minus construction-5",[f for f in FEATURES21 if f not in CONSTRUCTION5],None,"drop construction scale features")
add_check("log transform","21 log1p",FEATURES21,"log1p","log compression of scale")
add_check("standard scaling","21 standardized (source-fit)",FEATURES21,"standard","source-fit zscore")
add_check("robust scaling","21 robust-scaled (source-fit)",FEATURES21,"robust","source-fit median/IQR")
add_check("quantile normalization","21 quantile-normal (source-fit)",FEATURES21,"quantile","rank->normal mapping")
add_check("whitening","21 standardized+PCA-whiten (source-fit)",FEATURES21,"whiten","decorrelate + unit variance")
add_check("PCA projection","top-10 PCs (source-fit)",FEATURES21,"pca","linear dim reduction")

rob_df=pd.DataFrame(checks)
# threshold sensitivity (validation Youden vs fixed 0.5) reported as an observation row, AUC unaffected
rob_df["conclusion"]=np.where(rob_df["lodo_min_roc"]>=0.75,"transfer holds",
                       np.where(rob_df["lodo_min_roc"]>=0.55,"weak/partial transfer","transfer fails (near/under chance)"))
save_table(rob_df,"robustness_checks_diagnostic.csv")

# unavailable checks -> MISSING markers, recorded explicitly in the table too
unavail=[("covariate reweighting","importance/covariate reweighting module"),
         ("lightweight alignment","CORAL/feature-alignment module"),
         ("packet-sequence models","sequence model + per-flow packet arrays for all datasets (USBVPN lacks packet arrays)")]
for name,need in unavail:
    mark_missing("8", f"robustness check '{name}' not reproducible", need)
ua_df=pd.DataFrame([{"check":n,"model":"N/A","features":"N/A","train_domains":"N/A","val_domains":"N/A",
    "test_domains":"N/A","roc_auc_iscx":None,"roc_auc_usbvpn":None,"roc_auc_vnat":None,
    "lodo_mean_roc":None,"lodo_min_roc":None,"lodo_mean_pr":None,"lodo_min_pr":None,
    "observation":"component not present in project","conclusion":"MISSING / NEEDS MANUAL INPUT"} for n,_ in unavail])
rob_full=pd.concat([rob_df,ua_df],ignore_index=True)
save_table(rob_full,"robustness_checks_diagnostic.csv")  # overwrite with MISSING rows appended
print(rob_df[["check","lodo_mean_roc","lodo_min_roc","conclusion"]].to_string(index=False))
rob_full


  saved table: paper_audit_outputs\tables\robustness_checks_diagnostic.csv  (9x15)
  [MISSING/8] robustness check 'covariate reweighting' not reproducible  -> needs: importance/covariate reweighting module
  [MISSING/8] robustness check 'lightweight alignment' not reproducible  -> needs: CORAL/feature-alignment module
  [MISSING/8] robustness check 'packet-sequence models' not reproducible  -> needs: sequence model + per-flow packet arrays for all datasets (USBVPN lacks packet arrays)
  saved table: paper_audit_outputs\tables\robustness_checks_diagnostic.csv  (12x15)
                                        check  lodo_mean_roc  lodo_min_roc                         conclusion
                     baseline 21-feature LODO       0.582063      0.484982 transfer fails (near/under chance)
                         rate-feature removal       0.561532      0.475100 transfer fails (near/under chance)
feature pruning (no construction descriptors)       0.557356      0.454293 transfer fails (near/

,check,model,features,train_domains,val_domains,test_domains,roc_auc_iscx,roc_auc_usbvpn,roc_auc_vnat,lodo_mean_roc,lodo_min_roc,lodo_mean_pr,lodo_min_pr,observation,conclusion
0,baseline 21-feature LODO,"XGBoost(300,5,0.1)",21 safe_core_plus_temporal,two source datasets,source val split,held-out dataset,0.484982,0.548234,0.712974,0.582063,0.484982,0.193555,0.109557,reference LODO,transfer fails (near/under chance)
1,rate-feature removal,"XGBoost(300,5,0.1)","21 minus packet_rate,byte_rate",two source datasets,source val split,held-out dataset,0.4751,0.542599,0.666898,0.561532,0.4751,0.183727,0.097145,drop rate features,transfer fails (near/under chance)
2,feature pruning (no construction descriptors),"XGBoost(300,5,0.1)",21 minus construction-5,two source datasets,source val split,held-out dataset,0.454293,0.596264,0.62151,0.557356,0.454293,0.190653,0.088885,drop construction scale features,transfer fails (near/under chance)
3,log transform,"XGBoost(300,5,0.1)",21 log1p,two source datasets,source val split,held-out dataset,0.491999,0.516512,0.68138,0.563297,0.491999,0.180972,0.088921,log compression of scale,transfer fails (near/under chance)
4,standard scaling,"XGBoost(300,5,0.1)",21 standardized (source-fit),two source datasets,source val split,held-out dataset,0.491451,0.553081,0.692484,0.579005,0.491451,0.199887,0.093929,source-fit zscore,transfer fails (near/under chance)
5,robust scaling,"XGBoost(300,5,0.1)",21 robust-scaled (source-fit),two source datasets,source val split,held-out dataset,0.467712,0.52562,0.683175,0.558836,0.467712,0.184107,0.09796,source-fit median/IQR,transfer fails (near/under chance)
6,quantile normalization,"XGBoost(300,5,0.1)",21 quantile-normal (source-fit),two source datasets,source val split,held-out dataset,0.47264,0.480304,0.711637,0.55486,0.47264,0.171351,0.094031,rank->normal mapping,transfer fails (near/under chance)
7,whitening,"XGBoost(300,5,0.1)",21 standardized+PCA-whiten (source-fit),two source datasets,source val split,held-out dataset,0.453685,0.25199,0.571739,0.425805,0.25199,0.137252,0.061167,decorrelate + unit variance,transfer fails (near/under chance)
8,PCA projection,"XGBoost(300,5,0.1)",top-10 PCs (source-fit),two source datasets,source val split,held-out dataset,0.435126,0.350798,0.554614,0.446846,0.350798,0.148301,0.082144,linear dim reduction,transfer fails (near/under chance)
9,covariate reweighting,N/A,N/A,N/A,N/A,N/A,None,None,None,None,None,None,None,component not present in project,MISSING / NEEDS MANUAL INPUT


## 9. Publication-quality figures (300 DPI)

In [18]:

# All figures above were saved at dpi=300 via save_fig. The publication set required:
PUB=["intra_vs_cross_auc.png","construction_scale_boxplots.png",
     "dataset_fingerprinting_confusion.png","smd_heatmap.png",
     "preprocessing_sensitivity_heatmap.png"]
present=[p for p in PUB if (FIG/p).exists()]
missing_pub=[p for p in PUB if not (FIG/p).exists()]
for p in missing_pub: mark_missing("9", f"publication figure {p} not generated", "upstream section to succeed")
print("Publication figures present (300 DPI):")
for p in present: print("  -", p)
# also emit a dedicated intra-vs-cross PR figure if not already separate
fig,ax=plt.subplots(figsize=(7,5)); x=np.arange(len(DATASETS)); w=0.38
ax.bar(x-w/2,[intra_test_prauc.get(d,np.nan) for d in DATASETS],w,label="intra")
ax.bar(x+w/2,[lodo_pr.get(d,np.nan) for d in DATASETS],w,label="cross (LODO)")
ax.set(title="Intra vs Cross PR-AUC",xticks=x); ax.set_xticklabels([DS_LABEL[d] for d in DATASETS]); ax.legend()
save_fig(fig,"intra_vs_cross_pr_auc.png")


Publication figures present (300 DPI):
  - intra_vs_cross_auc.png
  - construction_scale_boxplots.png
  - dataset_fingerprinting_confusion.png
  - smd_heatmap.png
  - preprocessing_sensitivity_heatmap.png
  saved figure: paper_audit_outputs\figures\intra_vs_cross_pr_auc.png  (dpi=300)


WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/paper_audit_outputs/figures/intra_vs_cross_pr_auc.png')

## 10. Final summary + checklist

In [19]:

summary={
 "intra_roc_auc":{d:intra_test_auc.get(d) for d in DATASETS},
 "intra_pr_auc":{d:intra_test_prauc.get(d) for d in DATASETS},
 "lodo_roc_auc":lodo_roc,
 "lodo_pr_auc":lodo_pr,
 "lodo_mean_roc":float(np.nanmean(list(lodo_roc.values()))),
 "lodo_min_roc":float(np.nanmin(list(lodo_roc.values()))),
 "fingerprint_macro_auc":{ "A_construction5":fpA["macro_auc"], "B_full21":fpB["macro_auc"]},
 "n_verified_unstable_features":int((inst_df.instability_type=="verified raw-space direction instability").sum()),
 "n_rankbased_unstable_features":int((inst_df.instability_type=="rank-based direction instability").sum()),
 "n_stable_features":int((inst_df.instability_type=="stable feature").sum()),
 "n_missing_items":len(MISSING),
}
save_metric(summary,"nb2_final_summary.json")
save_table(pd.DataFrame(MISSING) if MISSING else pd.DataFrame([{"section":"-","what":"none","needed":"-"}]),
           "nb2_missing_items.csv")
print(json.dumps(summary,indent=2,default=str))
print("\nMISSING items:",len(MISSING))
for m in MISSING: print("  -",m["section"],m["what"])


  saved metric: paper_audit_outputs\metrics\nb2_final_summary.json
  saved table: paper_audit_outputs\tables\nb2_missing_items.csv  (5x3)
{
  "intra_roc_auc": {
    "iscx": 0.9834439568227662,
    "usbvpn": 0.9824307500778089,
    "vnat": 1.0
  },
  "intra_pr_auc": {
    "iscx": 0.9743986513792073,
    "usbvpn": 0.9986982288050603,
    "vnat": 1.0
  },
  "lodo_roc_auc": {
    "iscx": 0.484981871636966,
    "usbvpn": 0.5482335645763357,
    "vnat": 0.7129736368407914
  },
  "lodo_pr_auc": {
    "iscx": 0.28348325206400693,
    "usbvpn": 0.18762436199912047,
    "vnat": 0.1095566093840004
  },
  "lodo_mean_roc": 0.5820630243513644,
  "lodo_min_roc": 0.484981871636966,
  "fingerprint_macro_auc": {
    "A_construction5": 0.9854574476185486,
    "B_full21": 0.9993161668163092
  },
  "n_verified_unstable_features": 16,
  "n_rankbased_unstable_features": 5,
  "n_stable_features": 0,
  "n_missing_items": 5
}

MISSING items: 5
  - 7 covariate reweighting transform not available as reusable comp

In [20]:

# --- write checklist markdown ---
lines=[]
lines.append("# 02 — Model Transfer & Structural Diagnostics — Checklist\n")
lines.append(f"_Generated: {datetime.now().isoformat(timespec='seconds')}_\n")
lines.append("## Data source\n")
lines.append("- 21-feature matrix: `artifacts/clean_pipeline/features_{train,val,test}.parquet` (all 3 datasets, capture-level split).\n")
lines.append("- Label encoding: 1=VPN, 0=nonVPN. inf->NaN->0.0 cleaning applied.\n")
lines.append("- Capture leakage across splits: ZERO (asserted).\n\n")
lines.append("## Intra-dataset ROC-AUC / PR-AUC\n")
for d in DATASETS:
    lines.append(f"- {DS_LABEL[d]}: ROC-AUC={intra_test_auc.get(d)}, PR-AUC={intra_test_prauc.get(d)}\n")
lines.append("\n## LODO ROC-AUC / PR-AUC\n")
for d in DATASETS:
    lines.append(f"- target {DS_LABEL[d]}: ROC-AUC={lodo_roc.get(d)}, PR-AUC={lodo_pr.get(d)}\n")
lines.append(f"- LODO mean ROC-AUC={summary['lodo_mean_roc']:.5f}, min={summary['lodo_min_roc']:.5f}\n\n")
lines.append("## Dataset fingerprinting macro-AUC\n")
lines.append(f"- construction-5: {fpA['macro_auc']:.5f}\n- full-21: {fpB['macro_auc']:.5f}\n\n")
lines.append("## Feature instability\n")
lines.append(f"- verified raw-space SMD sign-flip: {summary['n_verified_unstable_features']}\n")
lines.append(f"- rank-based instability: {summary['n_rankbased_unstable_features']}\n")
lines.append(f"- stable: {summary['n_stable_features']}\n\n")
lines.append("## MISSING / NEEDS MANUAL INPUT\n")
if MISSING:
    for m in MISSING: lines.append(f"- [{m['section']}] {m['what']} -> needs: {m['needed']}\n")
else:
    lines.append("- none\n")
lines.append("\n## Generated files\n")
for g in sorted(set(GENERATED)): lines.append(f"- {g}\n")
p=LOG/"02_model_transfer_structural_diagnostics_checklist.md"
p.write_text("".join(lines),encoding="utf-8")
print("checklist written:",p.relative_to(ROOT))
print("total generated artifacts:",len(set(GENERATED)))


checklist written: paper_audit_outputs\logs\02_model_transfer_structural_diagnostics_checklist.md
total generated artifacts: 28
